# Đóng gói image Docker trên Colab (không cần Docker)

Dựng trên máy Colab đúng cây thư mục của image (`/code`, `/opt/venv`, `/opt/python`), chạy thử **chính** `predict.sh` trên GPU,
rồi ghép thành image bằng `crane` (không cần Docker daemon) trên nền `vllm/vllm-openai:v0.10.1.1`.
Kết quả ghi lên Drive `zaloai/`: `zaic2026_datamate_<team>.tar.gz` và `.md5`; ô 5 in kích thước `.tar` chưa nén.

Cần: runtime **A100**, Drive còn trống ≥ 40 GB, `datamate.zip` và `datamate_code.zip` mới nhất trong `zaloai/`.
Điền `TEAM` ở ô 1 rồi chạy lần lượt các ô (~1,5 giờ). Xong thì Runtime → Disconnect and delete runtime.

In [ ]:
# 1. Drive, mã nguồn và tập train vào /code (đúng đường dẫn trong image)
TEAM = ''  # tên đội: chữ thường, không dấu cách
assert TEAM and TEAM == TEAM.lower() and ' ' not in TEAM, 'điền TEAM'
from google.colab import drive
drive.mount('/content/drive')
import glob, os, shutil, zipfile
os.chdir('/content')
!df -h /content /content/drive | tail -2
Z = sorted(glob.glob('/content/drive/MyDrive/datamate.zip') + glob.glob('/content/drive/MyDrive/*/datamate.zip'))[0]
C = max(glob.glob('/content/drive/MyDrive/zaloai/datamate_code*.zip'), key=os.path.getmtime)
shutil.rmtree('/code', ignore_errors=True); shutil.rmtree('/content/dm', ignore_errors=True)
zipfile.ZipFile(Z).extractall('/content/dm')  # tập train vào image; public chỉ để chạy thử
zipfile.ZipFile(C).extractall('/code'); print('code from', C, os.path.getsize(C), 'bytes')
shutil.copytree('/content/dm/data/train', '/code/data/train')
for f in ('predict.sh', 'README.md', 'requirements.txt', 'training_code/data/bank.jsonl', 'data/train/train/questions.jsonl'):
    assert os.path.exists(f'/code/{f}'), f
!chmod +x /code/predict.sh

In [ ]:
# 2. Môi trường suy luận /opt/venv (Python 3.12 do uv quản lý, đặt ở /opt/python) + Qwen3.5-9B đúng revision khai báo
import subprocess
REV = 'c202236235762e1c871ad0ccb60c8ee5ba337b9a'
dl = subprocess.Popen(['python', '-c', f"from huggingface_hub import snapshot_download; snapshot_download('Qwen/Qwen3.5-9B', revision='{REV}', local_dir='/code/models/model')"],
                      stdout=open('/content/dl.log', 'w'), stderr=subprocess.STDOUT)
!pip -q install uv
!rm -rf /opt/venv /opt/python && UV_PYTHON_INSTALL_DIR=/opt/python UV_PYTHON_PREFERENCE=only-managed uv venv -q -p 3.12 /opt/venv && uv pip install -q -p /opt/venv/bin/python -r /code/requirements.txt 2>&1 | tail -3
!readlink -f /opt/venv/bin/python
!/opt/venv/bin/python -c "import torch, vllm, sys; assert torch.version.cuda.startswith('12.8'), torch.version.cuda; print('vllm', vllm.__version__, 'torch', torch.__version__, 'python', sys.version.split()[0])"
dl.wait(); print('download rc', dl.returncode); assert dl.returncode == 0, open('/content/dl.log').read()[-2000:]
!rm -rf /code/models/model/.cache && du -sh /code/models/model /opt/venv /opt/python

In [ ]:
# 3. data_manifest.json, rồi chạy thử CHÍNH predict.sh trên 30 câu public (LIMIT chỉ dùng khi thử; BTC chạy không có nó)
!cd /code && /opt/venv/bin/python -X utf8 -m training_code.manifest
!rm -rf /content/ptest && DATA=/content/dm/data/public_test/public_test RESULT=/content/ptest LIMIT=30 bash /code/predict.sh > /content/ptest.log 2>&1; tail -3 /content/ptest.log
!pkill -f vllm.entrypoints; wc -l /content/ptest/submission.jsonl /content/ptest/logs/log.jsonl
import json
r = json.loads(open('/content/ptest/logs/log.jsonl', encoding='utf-8').readline())
print({k: (str(v)[:80] if v is not None else None) for k, v in r.items() if k in ('id', 'answer', 'elapsed_ms', 'thinking', 'code', 'output')})
assert all(k in r for k in ('id', 'answer', 'elapsed_ms', 'thinking'))
assert sum(1 for _ in open('/content/ptest/submission.jsonl')) == 30

In [ ]:
# 4. Ghép image: nền vllm/vllm-openai:v0.10.1.1 + một lớp chứa /code, /opt/venv, /opt/python (crane, không cần Docker; ~30 phút)
!find /code -name __pycache__ -prune -exec rm -rf {} +
!tar --numeric-owner -C / -cf /content/layer.tar code opt/venv opt/python && ls -la /content/layer.tar
!curl -sL https://github.com/google/go-containerregistry/releases/latest/download/go-containerregistry_Linux_x86_64.tar.gz | tar -xz -C /usr/local/bin crane && crane version
IMAGE = f'zaic2026_datamate_{TEAM}:final'
!crane mutate vllm/vllm-openai:v0.10.1.1 --platform linux/amd64 --append /content/layer.tar --entrypoint bash --cmd /code/predict.sh -w /code -e HF_HUB_OFFLINE=1 -e TRANSFORMERS_OFFLINE=1 -t {IMAGE} -o /content/image.tar
!rm -f /content/layer.tar; ls -la /content/image.tar

In [ ]:
# 5. Kiểm tra image: tag, kiến trúc, entrypoint/cmd, đủ thư mục; kích thước .tar chưa nén (giới hạn 100 GB)
import json, tarfile
t = tarfile.open('/content/image.tar')
m = json.load(t.extractfile('manifest.json'))
cfg = json.load(t.extractfile(m[0]['Config']))
c = cfg['config']
print(m[0]['RepoTags'], len(m[0]['Layers']), 'layers', cfg['os'], cfg['architecture'])
print('entrypoint', c.get('Entrypoint'), 'cmd', c.get('Cmd'), 'workdir', c.get('WorkingDir'), [e for e in c['Env'] if 'OFFLINE' in e])
assert cfg['architecture'] == 'amd64' and c.get('Entrypoint') == ['bash'] and c.get('Cmd') == ['/code/predict.sh']
SIZE = os.path.getsize('/content/image.tar'); print('uncompressed .tar bytes:', SIZE); assert SIZE < 100e9

In [ ]:
# 6. Nén, md5, chép lên Drive và kiểm lại bản trên Drive.
#    Sau đó: chia sẻ file .tar.gz ở chế độ "Anyone with the link", nộp form Solution Submission với link + dòng md5 + số byte ở ô 5.
NAME = f'zaic2026_datamate_{TEAM}.tar.gz'
OUT = '/content/drive/MyDrive/zaloai'
!apt-get -qq install -y pigz > /dev/null
!cd /content && pigz -p 8 -c image.tar > {NAME} && rm image.tar && md5sum {NAME} > {NAME}.md5 && cat {NAME}.md5
!cp /content/{NAME} /content/{NAME}.md5 {OUT}/ && cd {OUT} && md5sum -c {NAME}.md5 && ls -la {NAME} {NAME}.md5